# 05 — End-to-end RAG eval (Ragas + MLflow)

Run the full chain over the 40-question test set; compute Ragas faithfulness / answer_relevancy / context_precision / context_recall + custom refusal_accuracy on the adversarial subset; log to MLflow.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import json
import os
from dotenv import load_dotenv
load_dotenv(Path("../.env"))

from src.retrieval.embeddings import BGEEmbedder
from src.retrieval.vector_store import ChromaStore
from src.retrieval.bm25_index import BM25Index
from src.retrieval.reranker import BGEReranker
from src.generation.llm_client import build_llm
from src.chain import answer_question
from src.eval.ragas_eval import load_qa_set, run_chain_on_qa, run_ragas_eval
from src.eval.mlflow_logger import log_eval_run

In [ ]:
# Load everything once
embedder = BGEEmbedder()
vec_store = ChromaStore(collection="filings", persist_dir=Path(os.environ["CHROMA_DIR"]))
bm25 = BM25Index.load(Path(os.environ["BM25_PATH"]))
reranker = BGEReranker()
llm = build_llm()

PROCESSED = Path("../data/processed")
chunk_lookup = {}
for jsonl in PROCESSED.rglob("*.jsonl"):
    with jsonl.open(encoding="utf-8") as f:
        for line in f:
            c = json.loads(line)
            chunk_lookup[c["chunk_hash"]] = {
                "text": c["text"],
                "metadata": {"ticker": c["ticker"], "year": c["year"], "page": c["page"]},
            }

qa_set = load_qa_set(Path("../data/qa_test_set.jsonl"))
print(f"QA set: {len(qa_set)} questions; chunk_lookup: {len(chunk_lookup)} chunks")

In [ ]:
# Bound chain function for the eval pipeline
def chain_fn(question: str) -> dict:
    return answer_question(
        question,
        embedder=embedder,
        vector_store=vec_store,
        bm25=bm25,
        reranker=reranker,
        llm=llm,
        chunk_lookup=chunk_lookup,
    )

In [ ]:
# Execute the chain over the 40 questions (this is the slow step — ~10 min on free Groq)
rows = run_chain_on_qa(qa_set, chain_fn)
print(f"Completed {len(rows)} runs")

In [ ]:
# Ragas metrics + custom refusal_accuracy
metrics = run_ragas_eval(rows)
metrics

In [ ]:
# Log to MLflow tagged by retrieval config (hybrid + reranker + query_rewrite is the default)
run_id = log_eval_run(
    metrics,
    config={
        "retrieval": "hybrid+rerank",
        "query_rewriter": "on",
        "embedder": "BAAI/bge-small-en-v1.5",
        "reranker": "BAAI/bge-reranker-v2-m3",
        "llm": os.environ.get("LLM_PROVIDER", "groq"),
    },
)
print(f"MLflow run: {run_id}")

In [ ]:
# Persist a snapshot of the per-question results for inspection
import pandas as pd
df = pd.DataFrame(rows)
df.to_csv("../docs/end_to_end_eval_run.csv", index=False)
print("Wrote ../docs/end_to_end_eval_run.csv")